In [1]:
import pandas as pd
import numpy as np
import re
import joblib

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

In [6]:
email_df = pd.read_csv("Phishing_Email.csv")
sms_df = pd.read_csv("spam.csv", encoding='latin-1')
url_df = pd.read_csv("malicious_phish.csv")

In [7]:
print(email_df.head())

   Unnamed: 0                                         Email Text  \
0           0  re : 6 . 1100 , disc : uniformitarianism , re ...   
1           1  the other side of * galicismos * * galicismo *...   
2           2  re : equistar deal tickets are you still avail...   
3           3  \nHello I am your hot lil horny toy.\n    I am...   
4           4  software at incredibly low prices ( 86 % lower...   

       Email Type  
0      Safe Email  
1      Safe Email  
2      Safe Email  
3  Phishing Email  
4  Phishing Email  


In [8]:
print(sms_df.head())

     v1                                                 v2 Unnamed: 2  \
0   ham  Go until jurong point, crazy.. Available only ...        NaN   
1   ham                      Ok lar... Joking wif u oni...        NaN   
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...        NaN   
3   ham  U dun say so early hor... U c already then say...        NaN   
4   ham  Nah I don't think he goes to usf, he lives aro...        NaN   

  Unnamed: 3 Unnamed: 4  
0        NaN        NaN  
1        NaN        NaN  
2        NaN        NaN  
3        NaN        NaN  
4        NaN        NaN  


In [9]:
print(url_df.head())

                                                 url        type
0                                   br-icloud.com.br    phishing
1                mp3raid.com/music/krizz_kaliko.html      benign
2                    bopsecrets.org/rexroth/cr/1.htm      benign
3  http://www.garage-pirenne.be/index.php?option=...  defacement
4  http://adventure-nicaragua.net/index.php?optio...  defacement


In [10]:
sms_df = sms_df[['v1', 'v2']]

sms_df.columns = ['label', 'text']

sms_df['label'] = sms_df['label'].map({'ham':0, 'spam':1})

sms_df.head()

,label,text
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [11]:

email_df.columns
email_df = email_df[['Email Text', 'Email Type']]

email_df.columns = ['text', 'label']

email_df['label'] = email_df['label'].map({
    'Safe Email': 0,
    'Phishing Email': 1
})

email_df.head()

,text,label
0,"re : 6 . 1100 , disc : uniformitarianism , re ...",0
1,the other side of * galicismos * * galicismo *...,0
2,re : equistar deal tickets are you still avail...,0
3,\nHello I am your hot lil horny toy.\n I am...,1
4,software at incredibly low prices ( 86 % lower...,1


In [12]:
url_df.columns

Index(['url', 'type'], dtype='object')

In [13]:
url_df = url_df[['url', 'type']]

url_df['label'] = url_df['type'].map({
    'benign': 0,
    'phishing': 1,
    'defacement': 1,
    'malware': 1
})

url_df = url_df[['url', 'label']]

url_df.head()

,url,label
0,br-icloud.com.br,1
1,mp3raid.com/music/krizz_kaliko.html,0
2,bopsecrets.org/rexroth/cr/1.htm,0
3,http://www.garage-pirenne.be/index.php?option=...,1
4,http://adventure-nicaragua.net/index.php?optio...,1


In [14]:
print(email_df.isnull().sum())
print(sms_df.isnull().sum())
print(url_df.isnull().sum())

text     16
label     0
dtype: int64
label    0
text     0
dtype: int64
url      0
label    0
dtype: int64


In [15]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'\W', ' ', text)
    return text

email_df['text'] = email_df['text'].apply(clean_text)
sms_df['text'] = sms_df['text'].apply(clean_text)

In [16]:
text_df = pd.concat([
    email_df[['text','label']],
    sms_df[['text','label']]
])

print(text_df.shape)

(24222, 2)


In [17]:
vectorizer = TfidfVectorizer(max_features=5000)

X_text = vectorizer.fit_transform(text_df['text'])
y_text = text_df['label']

In [18]:
url_vectorizer = TfidfVectorizer(max_features=3000)

X_url = url_vectorizer.fit_transform(url_df['url'])
y_url = url_df['label']

In [19]:
X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(X_text, y_text, test_size=0.2)

model = RandomForestClassifier()
model.fit(X_train_t, y_train_t)

RandomForestClassifier()

In [20]:
from sklearn.linear_model import LogisticRegression

X_train_u, X_test_u, y_train_u, y_test_u = train_test_split(X_url, y_url, test_size=0.2)

url_model = LogisticRegression(max_iter=1000)
url_model.fit(X_train_u, y_train_u)

LogisticRegression(max_iter=1000)

In [21]:
y_pred_t = model.predict(X_test_t)
print("Text Accuracy:", accuracy_score(y_test_t, y_pred_t))

Text Accuracy: 0.9562435500515996


In [22]:
y_pred_u = url_model.predict(X_test_u)
print("URL Accuracy:", accuracy_score(y_test_u, y_pred_u))

URL Accuracy: 0.9384285813005321


In [23]:
def predict_phishing(text):

    text_clean = clean_text(text)

    text_vec = vectorizer.transform([text_clean])  
    url_vec = url_vectorizer.transform([text_clean])  
    
    text_pred = model.predict(text_vec)[0]  
    url_pred = url_model.predict(url_vec)[0]  

    score = (text_pred + url_pred) / 2  
    
    if score == 0:  
        return "🟢 Safe"  
    elif score < 1:  
        return "🟡 Suspicious"  
    else:  
        return "🔴 Phishing"

In [24]:
predict_phishing("Congratulations! You have won a lottery. Click here now!")

'🟡 Suspicious'

In [25]:
predict_phishing("Hello, let's meet tomorrow for class.")

'🟢 Safe'

In [26]:
def predict_with_confidence(text):
    text_clean = clean_text(text)
    
    text_vec = vectorizer.transform([text_clean])
    prob = model.predict_proba(text_vec)[0][1]
    
    return f"Phishing Probability: {prob:.2f}"

In [27]:
def highlight_words(text):
    suspicious_words = ['win', 'lottery', 'free', 'click', 'urgent']
    
    found = [word for word in suspicious_words if word in text.lower()]
    
    return found

In [28]:
def batch_predict(df):
    df['result'] = df['text'].apply(predict_phishing)
    return df

In [29]:
from sklearn.naive_bayes import MultinomialNB

nb_model = MultinomialNB()
nb_model.fit(X_train_t, y_train_t)

y_pred_nb = nb_model.predict(X_test_t)

from sklearn.metrics import accuracy_score
print("Naive Bayes Accuracy:", accuracy_score(y_test_t, y_pred_nb))

Naive Bayes Accuracy: 0.9438596491228071


In [30]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_t, y_train_t)

y_pred_lr = lr_model.predict(X_test_t)
print("Logistic Regression Accuracy:", accuracy_score(y_test_t, y_pred_lr))

Logistic Regression Accuracy: 0.9504643962848297


In [31]:
from sklearn.naive_bayes import MultinomialNB

url_nb = MultinomialNB()
url_nb.fit(X_train_u, y_train_u)

print("URL NB Accuracy:", accuracy_score(y_test_u, url_nb.predict(X_test_u)))

URL NB Accuracy: 0.8899100883759857


In [34]:
import joblib

joblib.dump(model, "text_model.pkl")
joblib.dump(url_model, "url_model.pkl")

joblib.dump(vectorizer, "text_vectorizer.pkl")
joblib.dump(url_vectorizer, "url_vectorizer.pkl")

print(" Models saved successfully!")

 Models saved successfully!
